# 02 — Train & Evaluate on Google Colab

**LineSight — Visual Detection of Product Assembly Errors**

This notebook fine-tunes the anomaly-detection model on MVTec LOCO and evaluates
it, covering the **Model Training (25%)** and **Evaluation Metrics (20%)** rubric
items. Run it on Colab with a **T4 GPU**: *Runtime → Change runtime type → T4 GPU*.

It reuses the repository's own training/evaluation code so what you train here is
exactly what the app and API run.

## 1. Get the code

Clone your team's GitHub repo (replace the URL), or upload the project folder.
If you already pushed LineSight to GitHub, just clone it.

In [ ]:
# Option A — clone your repo (recommended). Replace with your URL:
# !git clone https://github.com/<your-team>/LineSight.git
# %cd LineSight

# Option B — if you uploaded the project as a zip to Colab:
# !unzip -q linesight.zip && %cd ivp

import os
print('Working dir:', os.getcwd())

## 2. Install dependencies

Colab already has torch + torchvision + matplotlib. We add the rest.

In [ ]:
!pip install -q numpy Pillow
import torch
print('CUDA available:', torch.cuda.is_available(), '| device:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Get the dataset

MVTec LOCO requires a manual form download (no direct link). Two common options
on Colab:

**A. Mount Google Drive** (upload the extracted dataset to Drive once):

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # then point DATA_DIR at your Drive folder that holds the category folders:
# DATA_DIR = '/content/drive/MyDrive/mvtec_loco'

# B. Or upload + extract the tar directly in Colab:
# from google.colab import files; files.upload()   # pick the .tar
# !mkdir -p /content/mvtec_loco && tar -xf mvtec_loco_anomaly_detection.tar -C /content/mvtec_loco
DATA_DIR = '/content/mvtec_loco'
CATEGORY = 'screw_bag'

## 4. Point the project at the dataset

Every script reads `IVP_DATA_DIR` and `IVP_CATEGORY` from the environment.
Setting them here means no code edits.

In [ ]:
import os
os.environ['IVP_DATA_DIR'] = DATA_DIR
os.environ['IVP_CATEGORY'] = CATEGORY
os.environ['IVP_MODEL_BACKEND'] = 'autoencoder'
print('IVP_DATA_DIR =', os.environ['IVP_DATA_DIR'])
print('IVP_CATEGORY =', os.environ['IVP_CATEGORY'])

## 5. Build the manifest (dataset → CSV)

Scans the LOCO folders and writes `manifest_<category>.csv` with split + label
for each image. This is the data-integration step.

In [ ]:
!python -m src.training.prepare_data --category $IVP_CATEGORY

## 6. Train the autoencoder (good images only)

Trains the convolutional autoencoder on `train/good`, tracks train + validation
loss (saved as a **training curve**), then calibrates the PASS/FAIL threshold from
the good-validation reconstruction errors. ~minutes on a T4.

Bump `--epochs` for a sharper model; 30 is a good start.

In [ ]:
!python -m src.training.train_anomaly --category $IVP_CATEGORY --epochs 30

### View the training curve

The rubric asks for training curves showing no overfitting (val loss should track
train loss, not diverge upward).

In [ ]:
from IPython.display import Image as IPyImage
curve = f'artifacts/eval/training_curve_{CATEGORY}.png'
IPyImage(filename=curve)

## 7. Evaluate on the test set

Produces the confusion matrix, precision/recall/F1, accuracy and AUROC, plus a
confusion-matrix PNG and ROC-curve PNG. **Recall** is the key metric (a missed
defect is the costly error).

In [ ]:
!python -m src.training.evaluate --category $IVP_CATEGORY

In [ ]:
from IPython.display import Image as IPyImage
import os
cm = f'artifacts/eval/confusion_{CATEGORY}.png'
roc = f'artifacts/eval/roc_{CATEGORY}.png'
for p in (cm, roc):
    if os.path.exists(p):
        display(IPyImage(filename=p))

## 8. Try a single prediction (sanity check)

Loads the trained detector through the same factory the app uses, and runs it on
one test defect to confirm it flags an anomaly with a heatmap.

In [ ]:
import glob
import os
from src.models import load_detector
from src.utils.images import load_image

det = load_detector()  # picks the trained autoencoder via IVP_MODEL_BACKEND
print('Loaded model:', det.name, '| threshold:', round(det.threshold, 3))

defects = glob.glob(f'{DATA_DIR}/{CATEGORY}/test/structural_anomalies/**/*.png', recursive=True)
if defects:
    res = det.predict(load_image(defects[0]))
    print('Sample defect ->', 'ANOMALY' if res.is_anomaly else 'good',
          '| score:', round(res.score, 3))

## 9. Save the model to Google Drive (so you don't retrain)

The trained weights + calibration JSON live in `artifacts/models/`. Copy them to
Drive, then download for your local demo (`make app`).

In [ ]:
# !mkdir -p /content/drive/MyDrive/linesight_models
# !cp artifacts/models/autoencoder_$IVP_CATEGORY.* /content/drive/MyDrive/linesight_models/
!ls -la artifacts/models/

## Done

You now have: a manifest (data integration), a trained model with a training
curve, evaluation metrics with confusion matrix + ROC, and a working detector.
Download `artifacts/models/` into your local repo and run `make app` for the demo.